<a href="https://colab.research.google.com/github/tommypolpo/geron-hands_on_ML/blob/main/c9_ex10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Train a deep MLP on the CoverType dataset (you can load it using `sklearn.datasets.fetch_covtype()`). See if you can get over 93% accuracy on the test set by fine-tuning the hyperparameters, manually and/or using `RandomizedSearchCV`.

In [1]:
from sklearn.datasets import fetch_covtype

covtype = fetch_covtype()


The samples in this dataset correspond to 30×30m patches of forest in the US, collected for the task of predicting each patch's cover type, i.e. the dominant species of tree. There are seven covertypes, making this a **multiclass classification problem**. Each sample has 54 features, described on the dataset's homepage

Name / Data Type / Measurement / Description

Elevation / quantitative /meters / Elevation in meters

Aspect / quantitative / azimuth / Aspect in degrees azimuth

Slope / quantitative / degrees / Slope in degrees

Horizontal_Distance_To_Hydrology / quantitative / meters / Horz Dist to nearest surface water features

Vertical_Distance_To_Hydrology / quantitative / meters / Vert Dist to nearest surface water features

Horizontal_Distance_To_Roadways / quantitative / meters / Horz Dist to nearest roadway

Hillshade_9am / quantitative / 0 to 255 index / Hillshade index at 9am, summer solstice

Hillshade_Noon / quantitative / 0 to 255 index / Hillshade index at noon, summer soltice

Hillshade_3pm / quantitative / 0 to 255 index / Hillshade index at 3pm, summer solstice

Horizontal_Distance_To_Fire_Points / quantitative / meters / Horz Dist to nearest wildfire ignition points

Wilderness_Area (4 binary columns) / qualitative / 0 (absence) or 1 (presence) / Wilderness area designation

Soil_Type (40 binary columns) / qualitative / 0 (absence) or 1 (presence) / Soil Type designation

Cover_Type (7 types) / integer / 1 to 7 / Forest Cover Type designation

Class Labels
Spruce/Fir, Lodgepole Pine, Ponderosa Pine, Cottonwood/Willow, Aspen, Douglas-fir, Krummholz

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

X_train, X_test, y_train, y_test = train_test_split(covtype.data, covtype.target,
                                                      test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"The first 10 labels are: {y_train[:10]}")
print(f"The first instance is: {X_train[0]}")

X_train shape: (464809, 54)
y_train shape: (464809,)
The first 10 labels are: [1 1 2 2 1 1 2 2 1 2]
The first instance is: [3.289e+03 2.200e+01 1.900e+01 2.400e+02 9.300e+01 1.708e+03 2.050e+02
 1.960e+02 1.220e+02 2.598e+03 0.000e+00 0.000e+00 1.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 1.000e+00 0.000e+00 0.000e+00
 0.000e+00 0.000e+00 0.000e+00 0.000e+00 0.000e+00]


It is a good idea to rescale the features.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

mlp = MLPClassifier(hidden_layer_sizes=[50,50,50], early_stopping=True,
                    verbose=True, random_state=42)

pipeline = make_pipeline(StandardScaler(), mlp)

pipeline.fit(X_train, y_train)


Iteration 1, loss = 0.60090206
Validation score: 0.781416
Iteration 2, loss = 0.47893514
Validation score: 0.800628
Iteration 3, loss = 0.43984649
Validation score: 0.819303
Iteration 4, loss = 0.41506524
Validation score: 0.827887
Iteration 5, loss = 0.39837728
Validation score: 0.832017
Iteration 6, loss = 0.38252454
Validation score: 0.840429
Iteration 7, loss = 0.36962517
Validation score: 0.846023
Iteration 8, loss = 0.36010835
Validation score: 0.857684
Iteration 9, loss = 0.35042816
Validation score: 0.856716
Iteration 10, loss = 0.34183587
Validation score: 0.862611
Iteration 11, loss = 0.33427153
Validation score: 0.857275
Iteration 12, loss = 0.32828638
Validation score: 0.867645
Iteration 13, loss = 0.32099259
Validation score: 0.868269
Iteration 14, loss = 0.31618237
Validation score: 0.868140
Iteration 15, loss = 0.31226356
Validation score: 0.872916
Iteration 16, loss = 0.30835973
Validation score: 0.871560
Iteration 17, loss = 0.30400665
Validation score: 0.873798
Iterat

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('mlpclassifier',
                 MLPClassifier(early_stopping=True,
                               hidden_layer_sizes=[50, 50, 50], random_state=42,
                               verbose=True))])

To avoid a mess, scikit-learn suppresses most estimator output when running searches in parallel. That is, the verbose=True in a SearchCV or RandomizedSearchCV does NOT output the partial validation score of each submodel on each fold.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

mlp = MLPClassifier(early_stopping=True,
                    verbose=True, random_state=42, activation="relu")

pipeline = make_pipeline(StandardScaler(), mlp)

param_distributions = {
    "mlpclassifier__hidden_layer_sizes": [
        (50,),
        (100,),
        (50, 50),
        (100, 50),
        (100, 100),
        (100, 100, 100),
    ],
    "mlpclassifier__batch_size": [64, 128, 256, 512],
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    cv=3,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

search.fit(X_train, y_train)


Fitting 3 folds for each of 10 candidates, totalling 30 fits
Iteration 1, loss = 0.51881767
Validation score: 0.818463
Iteration 2, loss = 0.38844591
Validation score: 0.847163
Iteration 3, loss = 0.33522086
Validation score: 0.872873
Iteration 4, loss = 0.30392573
Validation score: 0.877649
Iteration 5, loss = 0.28170105
Validation score: 0.888277
Iteration 6, loss = 0.26480846
Validation score: 0.891805
Iteration 7, loss = 0.25325711
Validation score: 0.894322
Iteration 8, loss = 0.24289549
Validation score: 0.897679
Iteration 9, loss = 0.23534073
Validation score: 0.904348
Iteration 10, loss = 0.22860683
Validation score: 0.905252
Iteration 11, loss = 0.22256899
Validation score: 0.905897
Iteration 12, loss = 0.21781827
Validation score: 0.908974
Iteration 13, loss = 0.21285047
Validation score: 0.912029
Iteration 14, loss = 0.20828398
Validation score: 0.914804
Iteration 15, loss = 0.20521387
Validation score: 0.911060
Iteration 16, loss = 0.20212811
Validation score: 0.914804
Iter

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('standardscaler',
                                              StandardScaler()),
                                             ('mlpclassifier',
                                              MLPClassifier(early_stopping=True,
                                                            random_state=42,
                                                            verbose=True))]),
                   n_jobs=-1,
                   param_distributions={'mlpclassifier__batch_size': [64, 128,
                                                                      256,
                                                                      512],
                                        'mlpclassifier__hidden_layer_sizes': [(50,),
                                                                              (100,),
                                                                              (50,
                                                                               50),
                                                                              (100,
                                                                               50),
                                                                              (100,
                                                                               100),
                                                                              (100,
                                                                               100,
                                                                               100)]},
                   random_state=42, scoring='accuracy', verbose=2)